In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import glob
import os
from sklearn.model_selection import GroupKFold

base_path = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'

print("学習データを読み込んでいます...")
train_files = glob.glob(f'{base_path}/train/*__horizontal_well.csv')
train_list = []
for f in train_files:
    df = pd.read_csv(f)
    df['well_id'] = os.path.basename(f).split('__')[0]
    train_list.append(df)
train_df = pd.concat(train_list, ignore_index=True)

print("テストデータを読み込んでいます...")
test_files = glob.glob(f'{base_path}/test/*__horizontal_well.csv')
test_list = []
for f in test_files:
    df = pd.read_csv(f)
    well_id = os.path.basename(f).split('__')[0]
    df['well_id'] = well_id
    # sample_submissionのID（例：000d7d20_1442）と一致させるための列を作る
    df['id'] = well_id + '_' + df.index.astype(str)
    test_list.append(df)
test_df = pd.concat(test_list, ignore_index=True)

print("モデルの学習を開始します...")
features = ['MD', 'X', 'Y', 'Z', 'GR']
target = 'TVT'

# 欠損値の処理
train_df[features] = train_df[features].fillna(0)
test_df[features] = test_df[features].fillna(0)
train_df = train_df.dropna(subset=[target])

# GroupKFoldによるモデル学習
gkf = GroupKFold(n_splits=5)
models = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, train_df[target], train_df['well_id'])):
    X_tr, y_tr = train_df.iloc[train_idx][features], train_df.iloc[train_idx][target]
    X_va, y_va = train_df.iloc[val_idx][features], train_df.iloc[val_idx][target]
    
    model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    models.append(model)

print("予測を行っています...")
preds = np.zeros(len(test_df))
for model in models:
    preds += model.predict(test_df[features]) / len(models)

# テストデータに予測値を格納
test_df['predicted_tvt'] = preds

print("提出ファイルを作成しています...")
sub = pd.read_csv(f'{base_path}/sample_submission.csv')

# IDをキーにして、テストデータから予測値を安全に引っ張ってくる（マージ）
sub = sub.drop(columns=['tvt']).merge(test_df[['id', 'predicted_tvt']], on='id', how='left')

# 列名をフォーマット通り 'tvt' に変更
sub = sub.rename(columns={'predicted_tvt': 'tvt'})

# 万が一予測できなかった行は0.0で埋める
sub['tvt'] = sub['tvt'].fillna(0.0)

# ★重要：提出ファイル名は絶対に 'submission.csv' にする！
sub[['id', 'tvt']].to_csv('submission.csv', index=False)
print("完了しました！submission.csv が作成されました。")